In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("customer_churn.csv")
df.head()

,CustomerID,Age,Gender,Tenure,MonthlyCharges,Contract,PaymentMethod,TotalCharges,Churn
0,1,56,Female,68,147.58,Two year,Bank transfer,10052.03,No
1,2,69,Male,32,22.54,Month-to-month,Mailed check,686.78,No
2,3,46,Female,10,52.47,One year,Electronic check,537.88,No
3,4,32,Male,22,109.67,Month-to-month,Mailed check,2390.04,Yes
4,5,60,Female,54,130.98,Month-to-month,Credit card,7081.28,No


In [3]:
df.shape

(100000, 9)

In [4]:
# Drop CustomerID (not useful for ML)
df = df.drop("CustomerID", axis=1)

In [5]:
# Convert TotalCharges to numeric (some may be empty)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

C:\Users\Jai Shiv\AppData\Local\Temp\ipykernel_9820\423839204.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [6]:
# Map Churn to numeric
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

In [7]:
# Encode Gender (binary)
df["Gender"] = df["Gender"].astype("category").cat.codes

In [8]:
# One-hot encode multi-class columns
df = pd.get_dummies(df, columns=["Contract", "PaymentMethod"], drop_first=True)


In [9]:
# Check
print(df.head())

   Age  Gender  Tenure  MonthlyCharges  TotalCharges  Churn  \
0   56       0      68          147.58      10052.03      0   
1   69       1      32           22.54        686.78      0   
2   46       0      10           52.47        537.88      0   
3   32       1      22          109.67       2390.04      1   
4   60       0      54          130.98       7081.28      0   

   Contract_One year  Contract_Two year  PaymentMethod_Credit card  \
0              False               True                      False   
1              False              False                      False   
2               True              False                      False   
3              False              False                      False   
4              False              False                       True   

   PaymentMethod_Electronic check  PaymentMethod_Mailed check  
0                           False                       False  
1                           False                        True  
2       

In [10]:
# Split Data for Training and Testing

In [11]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [12]:
X_train = X_train[:2000]
y_train = y_train[:2000]

In [13]:
# Scale Features

In [14]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [15]:
# Train SVC Model

In [16]:
svc_model = SVC(kernel="rbf", probability=True)
svc_model.fit(X_train, y_train)

svc_pred = svc_model.predict(X_test)

print("SVC Accuracy:", accuracy_score(y_test, svc_pred))

SVC Accuracy: 0.7278666666666667


In [17]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)

print("Logistic Accuracy:", accuracy_score(y_test, log_pred))

Logistic Accuracy: 0.7234333333333334


In [18]:
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

Random Forest Accuracy: 0.7216666666666667


In [19]:
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

Random Forest Accuracy: 0.7225


In [20]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.1, 1, 10],
    "gamma": ["scale", 0.1, 0.01],
    "kernel": ["rbf", "linear"]
}

grid = GridSearchCV(SVC(probability=True), param_grid, cv=5)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Parameters: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
Best Score: 0.713


In [21]:
best_model = grid.best_estimator_

In [22]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = best_model.predict(X_test)

print("Final Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Final Accuracy: 0.7319333333333333
              precision    recall  f1-score   support

           0       0.76      0.88      0.81     19961
           1       0.65      0.43      0.52     10039

    accuracy                           0.73     30000
   macro avg       0.70      0.66      0.67     30000
weighted avg       0.72      0.73      0.72     30000



In [23]:
print(grid.best_params_)

{'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}


In [24]:
import pickle

pickle.dump(best_model, open("final1_model.pkl","wb"))
pickle.dump(scaler, open("scaler1.pkl","wb"))